In [1]:
# import libs
import scanpy as sc
import anndata as ad
import os
import pandas as pd
import numpy as np
from scipy.io import mmread
from scipy.sparse import csr_matrix
import sqlite3

In [2]:
env_path = "../.env"

In [3]:
# call paths from .env
if os.path.exists(env_path):
    with open(env_path) as f:
        os.environ.update(
            line.strip().split("=", 1) for line in f
            if line.strip() and not line.startswith("#")
        )

data_dir = os.getenv("DATA_DIR")
ensembl_dir = os.getenv("ENSEMBL_DIR")
raw_adata = os.getenv("RAW_ANNDATA_DIR")

In [4]:
# load data
barcodes = pd.read_csv(data_dir + "/Zhong_hs_dg_GSE119212/barcodes.tsv", header = None)
genes = genes = pd.read_csv(data_dir + "/Zhong_hs_dg_GSE119212/genes.tsv", sep = "\t", header = None)
X = csr_matrix(mmread(data_dir + '/Zhong_hs_dg_GSE119212/matrix.mtx').T)

In [5]:
# print shapes 
print(f"X shape: {X.shape}")
print(f"Barcode shape: {barcodes.shape}")
print(f"Genes shape: {genes.shape}")

X shape: (33128, 32738)
Barcode shape: (33128, 1)
Genes shape: (32738, 2)


In [6]:
# build anndata
adata = ad.AnnData(X)
adata.obs_names = barcodes
adata.var_names = genes[1]
adata.obs_names

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:924: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    [('AAACCTGAGGAGTTGC-1',), ('AAACCTGAGGGTCGAT-1',), ('AAACCTGAGTATCGAA-1',), ('AAACCTGAGTTAAGTG-1',), ('AAACCTGCAATGCCAT-1',)]

    Inferred to be: mixed

  names = self._prep_dim_index(names, "obs")


Index([('AAACCTGAGGAGTTGC-1',), ('AAACCTGAGGGTCGAT-1',),
       ('AAACCTGAGTATCGAA-1',), ('AAACCTGAGTTAAGTG-1',),
       ('AAACCTGCAATGCCAT-1',), ('AAACCTGCACCCTATC-1',),
       ('AAACCTGCAGTAAGCG-1',), ('AAACCTGTCACAGGCC-1',),
       ('AAACCTGTCCACTGGG-1',), ('AAACCTGTCCATGCTC-1',),
       ...
       ('TTTGTCAAGGGCACTA-8',), ('TTTGTCAGTAAGGATT-8',),
       ('TTTGTCAGTCTGCCAG-8',), ('TTTGTCAGTGATGATA-8',),
       ('TTTGTCATCAACGCTA-8',), ('TTTGTCATCAATACCG-8',),
       ('TTTGTCATCACCAGGC-8',), ('TTTGTCATCAGAAATG-8',),
       ('TTTGTCATCCTTAATC-8',), ('TTTGTCATCGCACTCT-8',)],
      dtype='object', length=33128)

In [7]:
# remove suffixes and unnecessary brackets 
adata.obs_names = pd.Index([item[0].split('-')[0] for item in adata.obs_names])
adata.obs_names

Index(['AAACCTGAGGAGTTGC', 'AAACCTGAGGGTCGAT', 'AAACCTGAGTATCGAA',
       'AAACCTGAGTTAAGTG', 'AAACCTGCAATGCCAT', 'AAACCTGCACCCTATC',
       'AAACCTGCAGTAAGCG', 'AAACCTGTCACAGGCC', 'AAACCTGTCCACTGGG',
       'AAACCTGTCCATGCTC',
       ...
       'TTTGTCAAGGGCACTA', 'TTTGTCAGTAAGGATT', 'TTTGTCAGTCTGCCAG',
       'TTTGTCAGTGATGATA', 'TTTGTCATCAACGCTA', 'TTTGTCATCAATACCG',
       'TTTGTCATCACCAGGC', 'TTTGTCATCAGAAATG', 'TTTGTCATCCTTAATC',
       'TTTGTCATCGCACTCT'],
      dtype='object', length=33128)

In [8]:
print(adata.var_names)

Index(['MIR1302-10', 'FAM138A', 'OR4F5', 'RP11-34P13.7', 'RP11-34P13.8',
       'AL627309.1', 'RP11-34P13.14', 'RP11-34P13.9', 'AP006222.2',
       'RP4-669L17.10',
       ...
       'KIR3DL2', 'AL590523.1', 'CT476828.1', 'PNRC2', 'SRSF10', 'AC145205.1',
       'BAGE5', 'CU459201.1', 'AC002321.2', 'AC002321.1'],
      dtype='object', length=32738)


In [9]:
# subset ensembl to protein coding genes only
human_gtf_path = ensembl_dir
connection_hs = sqlite3.connect(human_gtf_path)
df_hs = pd.read_sql_query("SELECT * FROM gene;", connection_hs)
df_protein_coding_hs = df_hs[df_hs['gene_biotype'] == 'protein_coding']
print(f"Number of protein coding genes: {len(df_protein_coding_hs)}")

Number of protein coding genes: 19966


In [10]:
# keep protein coding genes only
adata = adata[:, adata.var_names.isin(df_protein_coding_hs['gene_name'].tolist())].copy()
adata

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:1899: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 33128 × 17954

In [11]:
adata.obs['dataset_of_origin'] = 'Zhong'

In [12]:
# Save the anndata
adata.write_h5ad(os.path.join(raw_adata, 'Adata_zhong_hippo_raw.h5ad'))